# Sprawozdanie z projektu EDI — System rekomendacji książek (JoyBOOkers)

**Przedmiot:** Eksploracja Danych Internetowych (sem. VI)  
**Wydział Informatyki, Politechnika Białostocka**  
**Autorzy:** Kiryl Alishkevich, Damian Kulesza  
**Prowadząca:** dr inż. Urszula Kużelewska  
**Grupa PS:** 1  

---

Niniejszy notebook stanowi **kompletne sprawozdanie akademickie** opisujące cały pipeline projektu JoyBOOkers: od surowych danych z internetu, przez modele uczenia maszynowego, po aplikację webową. Tekst napisany jest zrozumiałym językiem — bez zbędnego żargonu, z konkretnymi liczbami z raportów w katalogu `reports/`.

> **Jak korzystać z tego dokumentu:** każdy rozdział zawiera 1–2 akapity opisu oraz wskazówkę, **jaki zrzut ekranu lub wykres** warto wstawić do finalnego PDF. Możesz wyeksportować ten notebook do PDF (*File → Export* lub `jupyter nbconvert --to pdf`).

## Spis treści

1. [Wprowadzenie i cel projektu](#1-wprowadzenie)
2. [Zbiory danych (DS1–DS4)](#2-zbiory-danych)
3. [Architektura systemu](#3-architektura)
4. [Pipeline danych](#4-pipeline-danych)
5. [Analiza eksploracyjna (EDA)](#5-eda)
6. [Przetwarzanie tekstu (NLP)](#6-nlp)
7. [Selekcja cech](#7-selekcja-cech)
8. [Miary podobieństwa](#8-miary-podobienstwa)
9. [Filtrowanie kolaboratywne (SVD)](#9-svd)
10. [Rekomendacje oparte na treści (TF-IDF)](#10-content)
11. [Grupowanie użytkowników (K-Means)](#11-kmeans)
12. [Analiza sentymentu](#12-sentyment)
13. [System hybrydowy i porównanie podejść](#13-hybryda)
14. [Aplikacja internetowa](#14-aplikacja)
15. [Baza danych i retrenowanie CF](#15-baza)
16. [Podsumowanie i wnioski](#16-podsumowanie)
17. [Bibliografia](#17-bibliografia)

---
<a id="1-wprowadzenie"></a>
## 1. Wprowadzenie i cel projektu

Platformy takie jak LubimyCzytać pomagają czytelnikom wybierać książki spośród setek tysięcy tytułów. Nasz projekt **JoyBOOkers** realizuje podobną ideę: analizuje oceny użytkowników, metadane książek i teksty recenzji, a następnie proponuje spersonalizowane rekomendacje. Projekt powstał w ramach pracowni EDI i obejmuje pełny cykl eksploracji danych internetowych — od pobrania plików CSV z Kaggle, przez czyszczenie i wizualizację, po uczenie modeli i prezentację wyników w aplikacji webowej.

Celem pracy było zbudowanie **czterech modułów analitycznych** (filtrowanie kolaboratywne, rekomendacje treściowe, segmentacja użytkowników, analiza sentymentu) oraz połączenie ich w **jeden system hybrydowy** z interfejsem użytkownika. Technologie: Python 3.12+, pandas, scikit-learn, biblioteka Surprise (SVD), FastAPI, szablony Jinja2 z HTMX, baza SQLite. Cały pipeline można uruchomić jedną komendą: `python scripts/setup_all.py`, a następnie `uvicorn app.main:app --reload`.

> 📷 **Zrzut ekranu do wstawienia:** strona główna aplikacji (`http://127.0.0.1:8000/`) — pokazuje, że projekt ma działający interfejs, nie tylko skrypty w tle.

---
<a id="2-zbiory-danych"></a>
## 2. Zbiory danych (DS1–DS4)

Zamiast łączyć wszystko w jedną gigantyczną tabelę, projekt korzysta z **czterech osobnych zbiorów**, z których każdy ma przypisaną rolę. To świadoma decyzja projektowa — pełny katalog Goodreads ma ponad 1,5 mln książek i nie zmieściłby się w pamięci RAM przy budowie macierzy TF-IDF.

| Zbiór | Źródło (Kaggle) | Po czyszczeniu | Do czego służy |
|-------|-----------------|----------------|----------------|
| **DS1** | Goodreads 2M (książki + oceny) | 1,55 mln książek, **235 484** ocen, **3 980** użytkowników | SVD (CF), K-Means, sygnały behawioralne |
| **DS2** | Goodreads 100k Books | **99 999** książek | Główny katalog: gatunki, autorzy, opisy → TF-IDF |
| **DS3** | Best Books Ever | **52 424** książek | Wzbogacenie DS2 (postacie, tagi) przez dopasowanie tytułu+autora |
| **DS4** | Amazon Books Reviews | **1 887 091** recenzji | Analiza sentymentu (osobny moduł NLP, bez łączenia z Goodreads) |

**DS1** to serce behawioralne projektu. Pliki ocen z Kaggle zawierają tytuł książki zamiast ID — dlatego zaimplementowano dopasowywanie tytułów (dokładne, „core”, fuzzy z biblioteki RapidFuzz, próg 88). Dzięki temu z 362 596 surowych ocen zostało **235 484** poprawnych par użytkownik–książka. Gęstość macierzy ocen wynosi zaledwie **0,12%** — typowa rzadkość dla systemów rekomendacji.

**DS2 i DS3** łączą się w jeden katalog treściowy (**149 342** książek) po kluczu `match_key` (znormalizowany tytuł + autor). Stąd budowana jest macierz TF-IDF do wyszukiwania podobnych książek. **DS4** to recenzje Amazon z etykietami sentymentu (≥4★ = pozytywny, ≤2★ = negatywny; neutralne 3★ usunięte). Służy wyłącznie do modułu `/sentiment` — nie wpływa na ranking rekomendacji.

> 📷 **Wykres/tabela do wstawienia:** schemat blokowy (4 prostokąty DS1–DS4 ze strzałkami do modeli). Można narysować w draw.io lub użyć diagramu z sekcji 3 poniżej.  
> 📊 **Opcjonalnie:** tabela powyżej jako Rys. 1 w PDF.

---
<a id="3-architektura"></a>
## 3. Architektura systemu

Repozytorium dzieli się na trzy warstwy. **`bookrec/`** to biblioteka danych i ML (bez interfejsu) — ładowanie CSV, czyszczenie, budowa cech, trening i ewaluacja modeli. **`app/`** to aplikacja FastAPI z szablonami HTML, serwisami biznesowymi i silnikami inferencji ML ładowanymi przy starcie serwera. **`scripts/`** to punkty wejścia CLI (`setup_all.py`, `run_data_pipeline.py`, `run_ml_pipeline.py`). Artefakty trafiają do `data/processed/`, a metryki JSON i wykresy EDA — do `reports/` (wersjonowane w git).

Przepływ danych wygląda następująco:

```
Kaggle CSV (data/raw/)
       ↓
  run_data_pipeline.py  →  analyze → preprocess → features → splits
       ↓
  run_ml_pipeline.py    →  SVD | TF-IDF | K-Means | Sentiment
       ↓
  skrypty hybrydy       →  cluster_affinity, genre_priors, Ridge weights
       ↓
  load_db.py            →  SQLite (aplikacja)
       ↓
  uvicorn app.main:app  →  interfejs webowy + API REST
```

Użytkownik końcowy nie uruchamia pipeline'u — widzi gotową aplikację z załadowanymi modelami. Panel **Analityka** (`/analytics`) czyta metryki bezpośrednio z folderu `reports/`, więc wyniki eksperymentów są widoczne bez ponownego treningu.

> 📷 **Zrzut ekranu:** strona `/analytics` — pokazuje metryki pipeline'u w interfejsie użytkownika.

---
<a id="4-pipeline-danych"></a>
## 4. Pipeline danych

Pipeline danych uruchamia się poleceniem `python scripts/run_data_pipeline.py --stages all` i przechodzi przez cztery etapy. **Analyze** — wczytuje surowe pliki i profiluje schematy kolumn (moduły `bookrec/ingest/ds1_goodreads_2m.py` … `ds4_amazon_reviews.py`). **Preprocess** — czyści dane: usuwa puste wiersze, duplikaty, błędne ID, normalizuje nazwy kolumn, waliduje zakresy liczbowe (rok 1000–3000, ocena 1–5). **Features** — buduje macierze cech: interakcje user×book, cechy klasteryzacyjne użytkowników, katalog treściowy DS2+DS3, korpus NLP z DS4. **Splits** — dzieli dane na zbiory treningowe i testowe (CF: 80/20 per user; NLP: stratyfikowany 80/10/10).

Kluczowe pliki wyjściowe: `data/processed/ds1/interactions_clean.parquet` (235 484 wierszy), `data/processed/features/clustering/user_features_scaled.parquet` (3 311 użytkowników), `data/processed/features/content/content_catalog.parquet` (149 342 książek), `data/processed/splits/cf_train.parquet` (189 404) i `cf_test.parquet` (46 080). Raporty JSON kopiowane są do `reports/data_pipeline/` i `reports/features/`. Czyszczenie DS1 ocen jest najtrudniejszym krokiem — eksport Kaggle używa tytułów zamiast ID książek, stąd moduł `bookrec/title_matching.py` z dopasowaniem fuzzy.

> 📊 **Tabela do wstawienia:** podsumowanie liczb po preprocessingu (patrz sekcja 2).  
> 📷 **Opcjonalnie:** fragment logu z terminala po `python scripts/run_data_pipeline.py --stages all`.

---
<a id="5-eda"></a>
## 5. Analiza eksploracyjna (EDA)

Analiza eksploracyjna została zaimplementowana w module `bookrec/eda.py` i generuje wykresy PNG zapisywane w `reports/eda/`. Badamy rozkład ocen użytkowników (1★–5★), najpopularniejsze książki i najaktywniejszych użytkowników, rozkład języków w katalogu, koncentrację u wydawców oraz rozkład lat publikacji. Wyniki pokazują, że oceny 4★ i 5★ wyraźnie dominują nad niskimi — użytkownicy Goodreads są raczej hojni w ocenach. Dominującym językiem katalogu jest angielski, a kilku wydawców posiada zdecydowanie najwięcej tytułów.

Wykryto też wartości odstające: najaktywniejszy użytkownik ma **2 231** ocen, podczas gdy **454** użytkowników ma tylko jedną. **28 592** książek ma dokładnie jedną ocenę — to problem cold start dla CF. Średnia ocena w zbiorze wynosi **3,80** na skali 1–5. Te obserwacje uzasadniają zastosowanie modeli latent-factor (SVD) zamiast prostych średnich oraz hybrydę z sygnałami treściowymi i popularności.

> 📊 **Wykresy do wstawienia (z `reports/eda/`):**  
> - `01_rating_distribution.png` — rozkład ocen (Rys. 2)  
> - `02_top_books.png` — top 20 książek po liczbie ocen (Rys. 3)  
> - `03_top_users.png` — top 20 użytkowników (Rys. 4)  
> - wykres języków, wydawców, lat publikacji — jeśli są w folderze (Rys. 5–7)

---
<a id="6-nlp"></a>
## 6. Przetwarzanie tekstu (NLP)

Teksty (opisy książek z DS2/DS3 oraz recenzje z DS4) przechodzą normalizację w module `bookrec/text_normalization.py`: usuwanie tagów HTML, normalizacja Unicode (NFKD), redukcja wielokrotnych spacji, opcjonalna konwersja na małe litery. Tokenizacja odbywa się regexem `[a-z0-9']+`. Dla rekomendacji treściowych budujemy reprezentację **TF-IDF** (Term Frequency – Inverse Document Frequency) — każdy dokument (książka) staje się wektorem liczb opisujących ważność słów. Wiersze macierzy są **L2-normalizowane**, dzięki czemu podobieństwo cosinusowe sprowadza się do iloczynu skalarnego.

Dla analizy sentymentu (DS4) stosujemy pipeline scikit-learn: `TfidfVectorizer` (max 20 000 cech, bigramy, `min_df=2`, `sublinear_tf=True`) + **regresja logistyczna** z `class_weight='balanced'` (klasa negatywna jest rzadsza). Recenzje Amazon nie są łączone z katalogiem Goodreads — moduł sentymentu działa niezależnie i służy do demonstracji klasyfikacji tekstu w ramach wymagań EDI.

> 📷 **Zrzut ekranu:** strona `/sentiment` z przykładową recenzją i wynikiem „pozytywny/negatywny" (Rys. 8).

---
<a id="7-selekcja-cech"></a>
## 7. Selekcja cech

Cechy użytkowników do K-Means (`bookrec/features/clustering.py`) opisują **zachowanie przy ocenianiu**: liczba ocen (`n_ratings`), średnia ocena (`mean_rating`), odchylenie standardowe (`std_rating`), rozpiętość ocen (`rating_range`) oraz trzy wskaźniki aktywności (niska/średnia/wysoka) wyznaczone z kwantyli 33% i 66% liczby ocen. Użytkownicy z mniej niż **3 ocenami** są wykluczeni — zostaje **3 311** osób. Cechy są standaryzowane (średnia 0, odchylenie 1) przed K-Means.

Dla rekomendacji treściowych cechami są **wektory TF-IDF** z czterech bloków tekstowych: autorzy (3 000 terminów), gatunki (500), tagi (0 w obecnym DS3) i treść opisu (5 000) — łącznie **8 500** wymiarów dla **149 342** książek. W modelu hybrydowym jako „cechy" wykorzystujemy pięć **sygnałów**: predykcja CF, podobieństwo content, affinitet klastrowy, popularność i zgodność gatunkowa. Ich wagi uczone są regresją Ridge lub ustalane ręcznie w zależności od liczby ocen użytkownika.

---
<a id="8-miary-podobienstwa"></a>
## 8. Miary podobieństwa

W projekcie stosujemy **dwa rodzaje podobieństwa**. Dla tekstu (TF-IDF) — **podobieństwo cosinusowe** między wektorami książek lub między profilem użytkownika (ważona średnia wektorów ocenionych książek) a kandydatami. Średnie cosinusowe sąsiedztwo wynosi **0,63**, a pokrycie gatunków w top-10 podobnych książek — **71%**, co potwierdza sensowność dopasowania semantycznego.

Dla filtrowacji kolaboratywnej podobieństwo jest **ukryte (implicit)** — model SVD rozkłada macierz ocen na 100 czynników latentnych i porównuje użytkowników oraz książki w tej przestrzeni. Jakość grupowania użytkowników mierzymy współczynnikiem **Silhouette** (wartość **0,47** dla k=3). Jakość predykcji ocen — **RMSE** i **MAE** na zbiorze testowym.

---
<a id="9-svd"></a>
## 9. Filtrowanie kolaboratywne (SVD)

Algorytm **SVD** (Singular Value Decomposition) z biblioteki **Surprise** uczymy na ocenach DS1. Parametry: `n_factors=100`, `n_epochs=20`, `lr_all=0.005`, `reg_all=0.02`. Zbiór treningowy: **189 404** interakcji (80%), testowy: **46 080** (20%, holdout per user). Model zapisujemy w `data/processed/models/collaborative/svd_model.pkl`. W aplikacji obsługuje go klasa `CollaborativeFilteringEngine` (`app/ml/collaborative.py`) — przewiduje ocenę dla pary użytkownik–książka i generuje ranking kandydatów.

**Wyniki na zbiorze testowym:** RMSE = **0,89**, MAE = **0,71** (skala 1–5). To oznacza średni błąd około jednej „gwiazdki" — akceptowalne przy tak rzadkiej macierzy. Metryki rankingowe są słabsze: **Precision@10 = 0,56%**, **Recall@10 = 0,66%** — przy ~49 tys. książek trafienie w top-10 jest trudne, mimo że regresja działa rozsądnie. CF jest dostępne tylko dla użytkowników DS1 z co najmniej 3 ocenami, którzy znajdują się w macierzy treningowej SVD.

> 📊 **Tabela do wstawienia:** RMSE, MAE, Precision@10, Recall@10 (Rys. 9 — tabela metryk CF).

---
<a id="10-content"></a>
## 10. Rekomendacje oparte na treści (TF-IDF)

Moduł content-based (`app/ml/content_based.py`, trening: `bookrec/ml/content/`) buduje sparse macierz TF-IDF z katalogu DS2 wzbogaconego o DS3. Metoda `similar_books` znajduje książki najbliższe w przestrzeni cosinusowej — działa **bez danych innych użytkowników**, więc nadaje się do nowych tytułów i użytkowników bez historii CF. Profil użytkownika (`build_user_vector`) to ważona średnia wektorów ocenionych książek (waga = ocena).

Jakość mierzymy pośrednio: **średnie cosinusowe sąsiedztwo = 0,63**, **pokrycie gatunków w top-10 = 71%** (na próbce 500 książek). Jako predyktor surowej oceny (w baseline hybrydy) content ma RMSE = **0,72** — słabiej niż CF, ale uzupełnia go tam, gdzie brak historii behawioralnej. W aplikacji strona `/books/similar` pokazuje czysto content-based podobieństwo bez profilu użytkownika.

> 📷 **Zrzut ekranu:** `/books/similar` — lista podobnych książek do wybranego tytułu (Rys. 10).  
> 📷 **Zrzut ekranu:** `/books/{id}` — karta książki z gatunkami i metadanymi (Rys. 11).

---
<a id="11-kmeans"></a>
## 11. Grupowanie użytkowników (K-Means)

Algorytm **K-Means** dzieli **3 311** użytkowników na **3 klastry**. Parametr k dobieramy metryką **Silhouette** (testowane k=3…12; najlepsze: **k=3**, Silhouette = **0,47**). Model: `data/processed/models/clustering/kmeans_model.joblib`. W runtime `UserClusteringEngine` (`app/ml/user_clustering.py`) ładuje model, standaryzuje cechy z listy ocen i przypisuje klaster — także dla nowego użytkownika po kilku ocenach (formularz `/clustering/predict`).

| Klaster | Użytkownicy | Śr. liczba ocen | Śr. ocena | Charakterystyka |
|---------|-------------|-----------------|-----------|------------------|
| **0** | 1 378 (42%) | 149 | 3,82 | Aktywni czytelnicy — dużo ocen, nieco niższa średnia |
| **1** | 611 (18%) | 6 | 4,14 | Okazjonalni, hojni w gwiazdkach |
| **2** | 1 322 (40%) | 20 | 4,02 | Umiarkowana aktywność |

Klastry nie służą tylko do raportu — budujemy z nich **cluster affinity** (top 200 książek per klaster wg `liczba_ocen × średnia_ocena`) i **priory gatunków** per klaster, używane w rekomendacjach hybrydowych i cold-start.

> 📊 **Wykres do wstawienia:** wizualizacja PCA klastrów (z `/clustering` lub `reports/ml/clustering/`) — Rys. 12.  
> 📷 **Zrzut ekranu:** strona `/clustering` z opisami klastrów i formularzem predykcji — Rys. 13.

---
<a id="12-sentyment"></a>
## 12. Analiza sentymentu

Moduł sentymentu trenowany jest na **DS4** (Amazon Reviews): **1 509 675** wierszy treningowych, **188 708** walidacyjnych/testowych. Pipeline: TF-IDF + regresja logistyczna. Na zbiorze testowym: **accuracy = 91,8%**, **F1 macro = 85,0%**. Klasa pozytywna dominuje (~87% próbek), stąd niższa precyzja klasy negatywnej (0,64) przy wysokim recall (0,91). Zastosowano `class_weight='balanced'`, aby ograniczyć bias w stronę większościowej klasy.

Analiza sentymentu jest **osobnym modułem analitycznym** — wynik nie wpływa bezpośrednio na ranking rekomendacji w hybrydzie. W aplikacji użytkownik wkleja tekst recenzji na stronie `/sentiment` i otrzymuje etykietę pozytywny/negatywny wraz z prawdopodobieństwem. Spełnia to wymóg EDI dotyczący klasyfikacji opinii tekstowych.

> 📷 **Zrzut ekranu:** `/sentiment` z przykładową recenzją negatywną i pozytywną — Rys. 14.

---
<a id="13-hybryda"></a>
## 13. System hybrydowy i porównanie podejść

Sercem projektu jest **hybrydowy silnik rekomendacji** (`app/ml/hybrid_scoring.py`). Łączy pięć sygnałów: **CF** (predykcja SVD), **content** (cosinus TF-IDF), **cluster** (popularność książek w klastrze użytkownika), **pop** (liczba ocen w bazie) i **genre** (zgodność gatunków z historią lub priory klastra). Kandydaci (do 2 500) pochodzą z sumy: top CF, sąsiedzi content z ocenionych książek, affinitet klastrowy i książki startowe.

Wagi sygnałów zależą od liczby ocen użytkownika. Przy **0 ocenach** (cold start): 30% cluster + 45% pop + 25% genre. Przy **≥10 ocenach**: 40% CF + 35% content + 15% cluster + 5% pop + 5% genre. Dla użytkowników z dostępnym CF i załadowanym modelem Ridge stosujemy **wagi nauczone regresją** (`train_hybrid_weights.py`) — dają znacznie lepsze wyniki niż pojedyncze podejścia.

### Porównanie podejść (RMSE na znormalizowanej skali ocen, próbka 8 000 par)

| Podejście | RMSE | MAE | Kiedy najlepsze |
|-----------|------|-----|------------------|
| Tylko CF (SVD) | 0,170 | 0,134 | Użytkownik DS1 z bogatą historią w macierzy treningowej |
| Tylko content (TF-IDF) | 0,724 | 0,677 | Brak historii CF; „podobne książki" |
| Tylko klaster | 0,668 | 0,610 | Cold start, segmentacja |
| Tylko popularność | 0,735 | 0,689 | Úplnie nowy użytkownik |
| Hybryda ręczna | 0,544 | 0,500 | Użytkownicy zarejestrowani (bez CF) |
| **Hybryda Ridge** | **0,154** | **0,118** | Użytkownik DS1 z CF dostępnym |

Wniosek: **żaden pojedynczy algorytm nie wystarcza** — CF jest silne behawioralnie, content ratuje cold start, klaster i gatunki stabilizują propozycje dla nowych użytkowników. Ridge łączy sygnały optymalnie. W aplikacji domyślny tryb to `auto`/`hybrid`; można wymusić `collaborative` lub `content`.

> 📷 **Zrzut ekranu:** `/recommendations` lub `/users/{id}/recommendations` — lista rekomendacji z podsumowaniem profilu (klaster, wagi, algorytm) — **Rys. 15 (najważniejszy!)**  
> 📊 **Tabela:** porównanie baseline'ów powyżej — Rys. 16.

---
<a id="14-aplikacja"></a>
## 14. Aplikacja internetowa

Aplikacja to serwer **FastAPI** z interfejsem **server-rendered** (szablony Jinja2 + **HTMX** do dynamicznych fragmentów bez przeładowania strony). Styl: Bootstrap 5, wykresy Chart.js. Przy starcie (`app/startup.py`) ładowane są wszystkie modele ML przez `MLModelRegistry`. Dokumentacja API: `/docs` (Swagger). Język interfejsu: **polski**.

### Mapa funkcjonalności

| Strona | Adres | Co robi | Jaki model/analiza |
|--------|-------|---------|---------------------|
| Strona główna | `/` | Przegląd projektu, skrót metryk | — |
| Katalog | `/books` | Wyszukiwanie, filtry (gatunek, ocena) | SQLite + DS2/DS3 |
| Książka | `/books/{id}` | Szczegóły, gatunki, oceny | Baza aplikacji |
| Podobne | `/books/similar` | Książki podobne treściowo | TF-IDF (content) |
| Rekomendacje | `/recommendations` | Formularz: ID użytkownika DS1 | Hybryda |
| Użytkownicy | `/users`, `/users/{id}` | Profile i oceny z DS1 | DS1 + SQLite |
| Moje konto | `/register`, `/login`, `/me` | Rejestracja, logowanie | Sesje |
| Moje oceny | `/me/ratings`, `/me/rate` | Wystawianie ocen (HTMX) | SQLite → opcjonalny CF retrain |
| Moje rekomendacje | `/me/recommendations` | Rekomendacje zalogowanego | Hybryda ręczna (bez CF) |
| Klasteryzacja | `/clustering` | PCA, opisy klastrów, predykcja | K-Means |
| Sentyment | `/sentiment` | Klasyfikacja tekstu recenzji | LogReg + TF-IDF (DS4) |
| Analityka | `/analytics` | Metryki pipeline, EDA, cechy | `reports/*.json` |
| Autorzy | `/authors` | Przeglądanie po autorze | SQLite |

### Przykładowe scenariusze użytkownika

**Scenariusz A — użytkownik DS1 z historią:** Wchodzi na `/users/42/recommendations`. System buduje profil (`UserProfileBuilder`), sprawdza dostępność CF, liczy pięć sygnałów, scala wagi (Ridge lub ręczne), zwraca top-N książek z krótkim wyjaśnieniem po polsku (`explanations.py`).

**Scenariusz B — nowy zarejestrowany użytkownik:** Rejestracja → `/me` → ocenia 2 książki fantasy przez `/me/rate`. System przypisuje klaster, włącza `genre_prior_active`, rekomendacje na `/me/recommendations` opierają się na content (50%) + cluster (20%) + pop (20%) + genre (10%) — bez CF, bo nowe ID nie jest w macierzy SVD.

**Scenariusz C — eksploracja bez logowania:** `/books` → wyszukanie „Harry Potter" → `/books/similar` → lista fantasy przez TF-IDF. Osobno `/sentiment` — wklejenie recenzji i sprawdzenie tonu.

### API REST (`/api/v1`)

Dla integracji zewnętrznej: `GET /books`, `POST /recommendations/for-user`, `POST /sentiment/predict`, `GET /users/{id}/profile`, `POST /ratings` i in. Pełna lista w `/docs`.

> 📷 **Zrzuty ekranu — zestaw minimum (Rys. 17–22):**  
> 1. `/` — strona główna  
> 2. `/books` — wyszukiwarka z filtrami  
> 3. `/me/recommendations` — rekomendacje zalogowanego użytkownika z opisem profilu  
> 4. `/clustering` — wizualizacja PCA  
> 5. `/analytics` — panel metryk  
> 6. `/docs` — dokumentacja Swagger  
> 7. `/me/rate` — formularz oceniania (HTMX)

---
<a id="15-baza"></a>
## 15. Baza danych i retrenowanie CF

Warstwa operacyjna aplikacji korzysta z **SQLite** (`data/joybookers.db`). Skrypt `scripts/load_db.py` ładuje katalog z `content_catalog.parquet` (domyślnie **20 000** książek), użytkowników DS1 i **50 000** ocen. Tabele: `books`, `users`, `ratings`, `recommendations` (audyt), `reviews`. Zarejestrowani użytkownicy mają `is_registered=True`; oceny z aplikacji mają `source='app'`.

Zaimplementowano **retrenowanie CF** po nowych ocenach z aplikacji (`app/ml/cf_retrain.py`): po **10 zmianach** ocen (konfigurowalne) scheduler w tle eksportuje oceny `app`, scala z `cf_train.parquet`, trenuje SVD na nowo i przeładowuje model (`MLModelRegistry.reload_cf()`). Ręczne uruchomienie: `python scripts/retrain_cf_from_db.py`. To pokazuje, że system nie jest statyczny — może adaptować się do nowych danych z UI.

> 📷 **Opcjonalnie:** zrzut struktury bazy (np. DB Browser for SQLite) lub log retrenowania z terminala.

---
<a id="16-podsumowanie"></a>
## 16. Podsumowanie i wnioski

Zrealizowano kompletny system rekomendacji książek obejmujący cztery zbiory danych internetowych, pipeline czyszczenia i feature engineering, cztery modele ML oraz hybrydę z aplikacją webową. Oczyszczono ponad 1,5 mln rekordów katalogowych i 235 tys. interakcji. Najlepszy wynik predykcji uzyskano dla modelu hybrydowego Ridge (RMSE = 0,154). Analiza sentymentu osiągnęła 92% accuracy. Użytkowników podzielono na 3 klastry behawioralne (Silhouette = 0,47).

**Wnioski:** (1) Łączenie wielu sygnałów jest konieczne — CF samo w sobie słabo rankuje (P@10 < 1%), ale dobrze przewiduje oceny. (2) Rozdzielenie ról DS1–DS4 pozwoliło zmieścić projekt w RAM 8–16 GB. (3) Aplikacja webowa realnie demonstruje wszystkie komponenty — od EDA po rekomendacje i sentyment. (4) Ograniczenia: sentyment z Amazon nie jest domenowo dopasowany do Goodreads; zarejestrowani użytkownicy nie korzystają z CF bez mapowania ID; rekomendacje bywają trafne, ale nie zawsze na pierwszych pozycjach rankingu.

**Kierunki rozwoju:** metryki NDCG@K, testy A/B w UI, pełniejsze mapowanie użytkowników app→CF, rozszerzenie katalogu SQLite ponad 20k książek.

---
<a id="17-bibliografia"></a>
## 17. Bibliografia

1. Aggarwal C. C., *Recommender Systems: The Textbook*, Springer, 2016.  
2. Ricci F., Rokach L., Shapira B., *Recommender Systems Handbook*, Springer, 2022.  
3. Koren Y., Bell R., Volinsky C., „Matrix Factorization Techniques for Recommender Systems”, *Computer*, vol. 42, no. 8, 2009.  
4. Manning C. D., Raghavan P., Schütze H., *Introduction to Information Retrieval*, Cambridge University Press, 2008.  
5. Pedregosa F. et al., „Scikit-learn: Machine Learning in Python”, *JMLR*, vol. 12, 2011.  
6. Hug N., „Surprise: A Python library for recommender systems”, *JOSS*, vol. 5, no. 52, 2020.  
7. Rousseeuw P. J., „Silhouettes: a graphical aid to the interpretation and validation of cluster analysis”, *J. Comput. Appl. Math.*, 1987.  
8. Goodreads Book Datasets — Kaggle (DS1, DS2, DS3).  
9. Amazon Product Reviews — Kaggle (DS4).  
10. FastAPI Documentation, https://fastapi.tiangolo.com/  
11. Dokumentacja projektu: `DOKUMENTACJA_PROJEKTU.md`, `ETAP_DANYCH_I_ML.md`.

---
## Załącznik A — Źródła metryk w repozytorium

| Metryka | Wartość | Plik |
|---------|---------|------|
| Interakcje (clean) | 235 484 | `reports/data_pipeline/preprocess_summary.json` |
| Użytkownicy DS1 | 3 980 | jw. |
| Gęstość macierzy | 0,12% | jw. |
| Średnia ocena | 3,80 | `reports/features/features_summary.json` |
| SVD RMSE (test) | 0,890 | `reports/ml/evaluation/evaluate_all.json` |
| SVD MAE (test) | 0,706 | jw. |
| Precision@10 | 0,0056 | jw. |
| Sentyment accuracy | 91,8% | jw. |
| Sentyment F1 macro | 85,0% | jw. |
| K-Means Silhouette | 0,470 | `reports/ml/clustering/train_report.json` |
| Ridge hybrid RMSE | 0,154 | `reports/ml/evaluation/hybrid/baseline_comparison.json` |
| Książki w TF-IDF | 149 342 | `evaluate_all.json` |

## Załącznik B — Lista rysunków do finalnego PDF

| Nr | Opis | Źródło |
|----|------|--------|
| Rys. 1 | Schemat architektury DS1–DS4 | diagram (draw.io / sekcja 3) |
| Rys. 2 | Rozkład ocen użytkowników | `reports/eda/01_rating_distribution.png` |
| Rys. 3 | Top książki po liczbie ocen | `reports/eda/02_top_books.png` |
| Rys. 4 | Top użytkownicy | `reports/eda/03_top_users.png` |
| Rys. 5–7 | Języki, wydawcy, lata publikacji | `reports/eda/` |
| Rys. 8 | Analiza sentymentu | zrzut `/sentiment` |
| Rys. 9 | Tabela metryk CF | tabela z sekcji 9 |
| Rys. 10 | Podobne książki | zrzut `/books/similar` |
| Rys. 11 | Karta książki | zrzut `/books/{id}` |
| Rys. 12 | PCA klastrów | `/clustering` lub raport ML |
| Rys. 13 | Panel klasteryzacji | zrzut `/clustering` |
| Rys. 14 | Sentyment — przykłady | zrzut `/sentiment` |
| Rys. 15 | Rekomendacje hybrydowe z profilem | zrzut `/recommendations` |
| Rys. 16 | Porównanie baseline'ów | tabela z sekcji 13 |
| Rys. 17–22 | Aplikacja — zestaw UI | zrzuty z sekcji 14 |

In [ ]:
# Opcjonalna komórka: wczytanie metryk z raportów projektu (uruchom po setup_all.py)
from pathlib import Path
import json

ROOT = Path(".").resolve()
EVAL = ROOT / "reports/ml/evaluation/evaluate_all.json"
HYBRID = ROOT / "reports/ml/evaluation/hybrid/baseline_comparison.json"

def load_json(path):
    if path.is_file():
        return json.loads(path.read_text(encoding="utf-8"))
    return {"_error": f"Brak pliku: {path}. Uruchom: python scripts/setup_all.py"}

eval_all = load_json(EVAL)
hybrid = load_json(HYBRID)

print("=== Metryki modeli ML ===")
if "collaborative" in eval_all:
    cf = eval_all["collaborative"]
    print(f"SVD  RMSE={cf['rmse']:.4f}  MAE={cf['mae']:.4f}  Precision@10={cf['ranking']['precision_at_k']:.4f}")
if "sentiment" in eval_all:
    s = eval_all["sentiment"]["metrics"]
    print(f"Sentyment  accuracy={s['accuracy']:.2%}  F1 macro={s['f1_macro']:.4f}")
if "clustering" in eval_all:
    c = eval_all["clustering"]
    print(f"K-Means  k={c['n_clusters']}  Silhouette={c['silhouette']:.4f}  users={c['n_users']}")

print("\n=== Porównanie podejść hybrydowych (RMSE) ===")
if "baselines" in hybrid:
    for name, m in hybrid["baselines"].items():
        print(f"  {name:25s}  RMSE={m['rmse']:.4f}  MAE={m['mae']:.4f}")